# DeepSeek-style Language Model from Scratch
This notebook builds a small DeepSeek-style language model (Multi-head Latent Attention + DeepSeekMoE with shared experts and auxiliary-loss-free load balancing) using the same dataset/training pipeline style as the GPT-from-scratch notebook.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# downloading the dataset

In [ ]:
import os
import urllib.request

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

In [ ]:
# checking the dataset
print(text_data[-99:])

In [ ]:
# checking the tokens after the tokenization with byte-pair encoding using tiktoken
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))

print("characters:", total_characters)
print("Tokens:", total_tokens)

# the dataloader applied on the dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader

# This tokenizes the text dataset into overlapping input/target chunks, same idea as the GPT dataset class

class DeepSeekDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # tokenize the entire text
        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        # use sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# this loads the data into the compatible dataloader format

def create_dataloader_v1(txt, batch_size=4, max_length=256,
                          stride=128, shuffle=True, drop_last=True,
                          num_workers=0):
    # intialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # create dataset
    dataset = DeepSeekDatasetV1(txt, tokenizer, max_length, stride)
    print(len(dataset))

    # create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

# Configuration of the DeepSeek model

In [ ]:
DEEPSEEK_CONFIG_124M = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 256,   # Context length
    "emb_dim": 768,          # Embedding dimension
    "n_heads": 12,           # Number of attention heads
    "n_layers": 12,          # Number of transformer layers
    "drop_rate": 0.1,        # Dropout rate
    "norm_eps": 1e-6,        # RMSNorm epsilon
    "rope_theta": 10000,     # RoPE base frequency

    # Multi-head Latent Attention (MLA) dims
    "qk_nope_head_dim": 32,  # per-head query/key dim WITHOUT rotary embedding
    "qk_rope_head_dim": 32,  # per-head query/key dim WITH rotary embedding (decoupled RoPE)
    "v_head_dim": 64,        # per-head value dim
    "q_lora_rank": 192,      # low-rank compression size for queries
    "kv_lora_rank": 128,     # low-rank compression size for keys/values (the KV "latent")

    # DeepSeekMoE
    "first_k_dense_layers": 2,  # the first few layers use a plain dense FFN, just like DeepSeek-V3
    "moe_inter_dim": 384,       # hidden size of each (small) routed/shared expert
    "num_experts": 8,           # number of routed experts
    "num_shared_experts": 1,    # number of experts that are always active for every token
    "num_experts_per_tok": 2,   # top-k routed experts activated per token
}

# Train/validation ratio

In [ ]:
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=DEEPSEEK_CONFIG_124M["context_length"],
    stride=DEEPSEEK_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=DEEPSEEK_CONFIG_124M["context_length"],
    stride=DEEPSEEK_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
# checking the shape of the train_loader and validation loader
# 256 is the context size and 2 is the batch size, left side is input and right side is target
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

In [ ]:
# Tracking the number of tokens in the training and validation sets
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Training tokens:", train_tokens)
print("Validation tokens:", val_tokens)
print("All tokens:", train_tokens + val_tokens)

# RMSNorm (DeepSeek uses RMSNorm instead of LayerNorm)

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(emb_dim))

    def forward(self, x):
        # root-mean-square normalize, then apply a learned per-channel scale (no shift/bias, unlike LayerNorm)
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight

# Rotary Positional Embeddings (RoPE)

In [ ]:
def precompute_rope_params(head_dim, theta_base=10000, context_length=4096):
    assert head_dim % 2 == 0, "head_dim must be even for RoPE"

    # compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2).float() / head_dim))

    # compute the rotation angles for every position
    positions = torch.arange(context_length)
    angles = positions[:, None] * inv_freq[None, :]  # (context_length, head_dim/2)

    # duplicate angles so they cover the full head_dim
    angles = torch.cat([angles, angles], dim=1)  # (context_length, head_dim)

    cos = torch.cos(angles)
    sin = torch.sin(angles)
    return cos, sin


def apply_rope(x, cos, sin):
    # x: (batch, heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "head_dim must be even for RoPE"

    # split x into two halves
    x1 = x[..., : head_dim // 2]
    x2 = x[..., head_dim // 2:]

    # match the sequence length and add batch/head dims for broadcasting
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # rotate the pairs and combine
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    return x_rotated.to(dtype=x.dtype)

# Multi-head Latent Attention (MLA)

In [ ]:
class MultiHeadLatentAttention(nn.Module):
    """
    DeepSeek's Multi-head Latent Attention.
    Instead of caching full-size keys/values per head, queries and keys/values are first
    compressed into a small low-rank "latent" vector, then up-projected back per head.
    Only a small slice of each head (qk_rope_head_dim) gets rotary embeddings applied
    (this is called "decoupled RoPE"); the rest (qk_nope_head_dim) does not use position info directly.
    """

    def __init__(self, d_in, num_heads, qk_nope_head_dim, qk_rope_head_dim, v_head_dim,
                 q_lora_rank, kv_lora_rank, context_length, dropout, rope_theta=10000):
        super().__init__()
        self.num_heads = num_heads
        self.qk_nope_head_dim = qk_nope_head_dim
        self.qk_rope_head_dim = qk_rope_head_dim
        self.qk_head_dim = qk_nope_head_dim + qk_rope_head_dim
        self.v_head_dim = v_head_dim
        self.kv_lora_rank = kv_lora_rank

        # ---- Query path: down-project to a small rank, then up-project per head ----
        self.q_down_proj = nn.Linear(d_in, q_lora_rank, bias=False)
        self.q_norm = RMSNorm(q_lora_rank)
        self.q_up_proj = nn.Linear(q_lora_rank, num_heads * self.qk_head_dim, bias=False)

        # ---- Key/Value path: a single down-projection produces the shared KV latent
        #      plus a small decoupled rotary key that is shared across all heads ----
        self.kv_down_proj = nn.Linear(d_in, kv_lora_rank + qk_rope_head_dim, bias=False)
        self.kv_norm = RMSNorm(kv_lora_rank)
        self.kv_up_proj = nn.Linear(kv_lora_rank, num_heads * (qk_nope_head_dim + v_head_dim), bias=False)

        self.out_proj = nn.Linear(num_heads * v_head_dim, d_in, bias=False)
        self.dropout = nn.Dropout(dropout)

        cos, sin = precompute_rope_params(qk_rope_head_dim, rope_theta, context_length)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, seq_len, _ = x.shape

        # ---- Queries ----
        q = self.q_up_proj(self.q_norm(self.q_down_proj(x)))
        q = q.view(b, seq_len, self.num_heads, self.qk_head_dim).transpose(1, 2)
        q_nope, q_rope = torch.split(q, [self.qk_nope_head_dim, self.qk_rope_head_dim], dim=-1)
        q_rope = apply_rope(q_rope, self.cos, self.sin)
        queries = torch.cat([q_nope, q_rope], dim=-1)  # (b, num_heads, seq_len, qk_head_dim)

        # ---- Keys / Values, decompressed from the shared latent ----
        kv = self.kv_down_proj(x)
        kv_latent, k_rope = torch.split(kv, [self.kv_lora_rank, self.qk_rope_head_dim], dim=-1)
        kv_latent = self.kv_norm(kv_latent)
        kv_up = self.kv_up_proj(kv_latent)
        kv_up = kv_up.view(b, seq_len, self.num_heads, self.qk_nope_head_dim + self.v_head_dim).transpose(1, 2)
        k_nope, values = torch.split(kv_up, [self.qk_nope_head_dim, self.v_head_dim], dim=-1)

        # the decoupled rotary key is a single shared vector, broadcast to every head
        k_rope = k_rope.view(b, seq_len, 1, self.qk_rope_head_dim).transpose(1, 2)
        k_rope = apply_rope(k_rope, self.cos, self.sin)
        k_rope = k_rope.expand(b, self.num_heads, seq_len, self.qk_rope_head_dim)

        keys = torch.cat([k_nope, k_rope], dim=-1)  # (b, num_heads, seq_len, qk_head_dim)

        # ---- scaled dot-product attention with a causal mask ----
        attn_scores = queries @ keys.transpose(2, 3)

        mask_bool = self.mask.bool()[:seq_len, :seq_len]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / self.qk_head_dim**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2).contiguous()
        context_vec = context_vec.view(b, seq_len, self.num_heads * self.v_head_dim)
        context_vec = self.out_proj(context_vec)

        return context_vec

# SwiGLU feed forward network (used for the dense layers and every expert)

In [ ]:
class SwiGLUFeedForward(nn.Module):
    def __init__(self, emb_dim, hidden_dim):
        super().__init__()
        self.gate_proj = nn.Linear(emb_dim, hidden_dim, bias=False)
        self.up_proj = nn.Linear(emb_dim, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, emb_dim, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

# DeepSeekMoE: fine-grained experts, shared experts, and auxiliary-loss-free load balancing

In [ ]:
class Expert(nn.Module):
    """A single routed/shared expert: a small SwiGLU feed forward network."""

    def __init__(self, emb_dim, moe_inter_dim):
        super().__init__()
        self.ffn = SwiGLUFeedForward(emb_dim, moe_inter_dim)

    def forward(self, x):
        return self.ffn(x)

In [ ]:
class DeepSeekMoE(nn.Module):
    """
    DeepSeek-style Mixture of Experts.

    - Uses a sigmoid "affinity" score per expert instead of a softmax (as in DeepSeek-V3).
    - A small number of "shared" experts process every token (they capture common knowledge),
      while a top-k subset of "routed" experts is selected per token for specialized knowledge.
    - Load balancing is done WITHOUT an auxiliary loss: a per-expert bias is nudged up or down
      after every training step depending on whether that expert was over- or under-used, and
      that bias only affects *which* experts get picked, never how much weight they receive.
    """

    def __init__(self, emb_dim, moe_inter_dim, num_experts, num_shared_experts,
                 num_experts_per_tok, bias_update_speed=0.001):
        super().__init__()
        self.num_experts = num_experts
        self.num_experts_per_tok = num_experts_per_tok
        self.bias_update_speed = bias_update_speed

        self.gate = nn.Linear(emb_dim, num_experts, bias=False)
        # the load-balancing bias is a buffer, not a trainable parameter - it is updated manually
        self.register_buffer("expert_bias", torch.zeros(num_experts))

        self.routed_experts = nn.ModuleList(
            [Expert(emb_dim, moe_inter_dim) for _ in range(num_experts)]
        )
        self.shared_experts = nn.ModuleList(
            [Expert(emb_dim, moe_inter_dim) for _ in range(num_shared_experts)]
        )

    def forward(self, x):
        b, seq_len, emb_dim = x.shape
        flat_x = x.view(-1, emb_dim)  # (b*seq_len, emb_dim)

        # sigmoid affinity scores for every (token, expert) pair
        scores = torch.sigmoid(self.gate(flat_x))          # (tokens, num_experts)
        routing_scores = scores + self.expert_bias          # bias only changes *selection*

        top_k_scores, top_k_indices = routing_scores.topk(self.num_experts_per_tok, dim=-1)
        # the actual gating weight uses the ORIGINAL (unbiased) affinity score, re-normalized
        top_k_gates = torch.gather(scores, dim=-1, index=top_k_indices)
        top_k_gates = top_k_gates / (top_k_gates.sum(dim=-1, keepdim=True) + 1e-9)

        gating_full = torch.zeros_like(scores)
        gating_full.scatter_(-1, top_k_indices, top_k_gates)

        final_output = torch.zeros_like(flat_x)

        # dispatch tokens to their selected routed experts
        for i, expert in enumerate(self.routed_experts):
            expert_mask = (top_k_indices == i).any(dim=-1)

            if expert_mask.any():
                expert_input = flat_x[expert_mask]
                expert_output = expert(expert_input)

                gating_scores = gating_full[expert_mask, i].unsqueeze(1)
                final_output[expert_mask] += expert_output * gating_scores

        # shared experts see every single token, no routing involved
        for expert in self.shared_experts:
            final_output = final_output + expert(flat_x)

        # update the load-balancing bias (no gradient): push down experts that are overloaded,
        # push up experts that are underused, so future routing decisions self-balance
        if self.training:
            with torch.no_grad():
                load = torch.bincount(top_k_indices.view(-1), minlength=self.num_experts).float()
                avg_load = load.mean()
                self.expert_bias += self.bias_update_speed * torch.sign(avg_load - load)

        return final_output.view(b, seq_len, emb_dim)

In [ ]:
# quick sanity check that the shapes work out
_test_moe = DeepSeekMoE(emb_dim=32, moe_inter_dim=64, num_experts=4, num_shared_experts=1, num_experts_per_tok=2)
_test_input = torch.randn(2, 5, 32)
_test_output = _test_moe(_test_input)
print("MoE output shape:", _test_output.shape)  # should match the input shape

# The DeepSeek Transformer block

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg, use_moe):
        super().__init__()
        self.att = MultiHeadLatentAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            qk_nope_head_dim=cfg["qk_nope_head_dim"],
            qk_rope_head_dim=cfg["qk_rope_head_dim"],
            v_head_dim=cfg["v_head_dim"],
            q_lora_rank=cfg["q_lora_rank"],
            kv_lora_rank=cfg["kv_lora_rank"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"],
            rope_theta=cfg["rope_theta"]
        )

        # the first few layers use a plain dense FFN, just like DeepSeek-V3 - only later
        # layers get the (much larger capacity) mixture-of-experts FFN
        if use_moe:
            self.ff = DeepSeekMoE(
                emb_dim=cfg["emb_dim"],
                moe_inter_dim=cfg["moe_inter_dim"],
                num_experts=cfg["num_experts"],
                num_shared_experts=cfg["num_shared_experts"],
                num_experts_per_tok=cfg["num_experts_per_tok"]
            )
        else:
            self.ff = SwiGLUFeedForward(cfg["emb_dim"], cfg["emb_dim"] * 4)

        self.norm1 = RMSNorm(cfg["emb_dim"], eps=cfg["norm_eps"])
        self.norm2 = RMSNorm(cfg["emb_dim"], eps=cfg["norm_eps"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for the attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        # Shortcut connection for the feed forward / MoE block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

# The DeepSeek model

In [ ]:
class DeepSeekModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        # note: unlike the GPT model, there is no separate learned position embedding table -
        # position information is injected inside attention via RoPE instead

        self.trf_blocks = nn.ModuleList([
            TransformerBlock(cfg, use_moe=(i >= cfg["first_k_dense_layers"]))
            for i in range(cfg["n_layers"])
        ])

        self.final_norm = RMSNorm(cfg["emb_dim"], eps=cfg["norm_eps"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        x = self.tok_emb(in_idx)
        x = self.drop_emb(x)
        for block in self.trf_blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

# loading the model

In [ ]:
torch.manual_seed(123)
model = DeepSeekModel(DEEPSEEK_CONFIG_124M)
model.eval();  # Disable dropout during inference

# Calculating the loss of the model

In [ ]:
# function for loss calculation of the model
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Note:
# Uncommenting the following lines will allow the code to run on Apple Silicon chips, if applicable,
# which is approximately 2x faster than on an Apple CPU (as measured on an M3 MacBook Air).
# However, the resulting loss values may be slightly different.

#if torch.cuda.is_available():
#    device = torch.device("cuda")
#elif torch.backends.mps.is_available():
#    device = torch.device("mps")
#else:
#    device = torch.device("cpu")
#
# print(f"Using {device} device.")


model.to(device)  # no assignment model = model.to(device) necessary for nn.Module classes


torch.manual_seed(123)  # For reproducibility due to the shuffling in the data loader

with torch.no_grad():  # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

# Generating a simple text

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):

        # Crop current context if it exceeds the supported context size
        # E.g., if the model supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)  # batch, n_tokens, vocab_size

        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

In [ ]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)  # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)  # remove batch dimension
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=DEEPSEEK_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

# Training Loop for LLM

In [ ]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                        eval_freq, eval_iter, start_context, tokenizer):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()  # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()  # Calculate loss gradients
            optimizer.step()  # Update model weights using loss gradients
            tokens_seen += input_batch.numel()  # Returns the total number of elements (or tokens) in the input_batch.
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Print a sample text after each epoch
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen

<div class="alert alert-block alert-info">

Step 1: Initialize lists to track losses and tokens seen

Step 2: Start the main training loop

Step 3: Reset loss gradients from previous batch iteration

Step 4: Calculate loss gradients

Step 5: Update model weights using loss gradients

Step 6: Optional evaluation step

Step 7: Print a sample text after each epoch

</div>

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [ ]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = DEEPSEEK_CONFIG_124M["context_length"]  # no pos_emb table to read the size from, unlike GPT
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()

In [ ]:
# Note:
# Uncomment the following code to calculate the execution time
import time
start_time = time.time()

torch.manual_seed(123)
model = DeepSeekModel(DEEPSEEK_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

# Note:
# Uncomment the following code to show the execution time
end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

#Plotting loss after training 10 epochs

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))  # only show integer labels on x-axis

    # Create a second x-axis for tokens seen
    ax2 = ax1.twiny()  # Create a second x-axis that shares the same y-axis
    ax2.plot(tokens_seen, train_losses, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()  # Adjust layout to make room
    plt.savefig("loss-plot.pdf")
    plt.show()

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

#Temperature scaling and top-k sampling

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):

    # For-loop is the same as before: Get logits, and only focus on last time step
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # New: Filter logits with top_k sampling
        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        # New: Apply temperature scaling
        if temperature > 0.0:
            logits = logits / temperature

            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if idx_next == eos_id:  # Stop generating early if end-of-sequence token is encountered and eos_id is specified
            break

        # Same as before: append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

In [ ]:
# implementing the generation after the temperature scaling and top-k sampling
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=15,
    context_size=DEEPSEEK_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

# LOADING AND SAVING MODEL WEIGHTS IN PYTORCH

In [ ]:
# saving the model
torch.save(model.state_dict(), "deepseek_model.pth")

In [ ]:
# loading the model
model = DeepSeekModel(DEEPSEEK_CONFIG_124M)
model.load_state_dict(torch.load("deepseek_model.pth", map_location=device))
model.to(device)
model.eval()